In [0]:
raw = "/Volumes/workspace/default/berka"
display(dbutils.fs.ls(raw))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/berka/account.csv,account.csv,155356,1791945884000
dbfs:/Volumes/workspace/default/berka/card.csv,card.csv,31588,1791945884000
dbfs:/Volumes/workspace/default/berka/client.csv,client.csv,94820,1791945884000
dbfs:/Volumes/workspace/default/berka/disp.csv,disp.csv,129716,1791945884000
dbfs:/Volumes/workspace/default/berka/district.csv,district.csv,6574,1791945884000
dbfs:/Volumes/workspace/default/berka/loan.csv,loan.csv,27037,1791945884000
dbfs:/Volumes/workspace/default/berka/order.csv,order.csv,273800,1791945884000
dbfs:/Volumes/workspace/default/berka/trans.csv,trans.csv,69406578,1791945904000


In [0]:
from pyspark.sql import functions as F
import os

bronze_base = "/Volumes/workspace/default/lakehouse/bronze"
files = sorted([f.path for f in dbutils.fs.ls(raw) if f.path.endswith(".csv")])
print(len(files), "files")
for f in files:
  name = os.path.splitext(os.path.basename(f))[0]
  print("loading ", name)

  df = spark.read.csv(f, header = True, sep = ";", inferSchema= False)
  df = df.withColumn("_ingestion_time", F.current_timestamp())
  df = df.withColumn("_source_file", F.col("_metadata.file_path"))

  df.write.format("delta").mode("overwrite").save(f"{bronze_base}/{name}")

  saved = spark.read.format("delta").load(f"{bronze_base}/{name}")
  print(name , saved.count(), "rows")

8 files
loading  account
account 4500 rows
loading  card
card 892 rows
loading  client
client 5369 rows
loading  disp
disp 5369 rows
loading  district
district 77 rows
loading  loan
loan 682 rows
loading  order
order 6471 rows
loading  trans
trans 1056320 rows


In [0]:
tables = [ "account","card","client","disp","district","loan","order","trans"]

for t in tables:
  d= spark.read.format("delta").load(f"{bronze_base}/{t}")
  cols = [c for c in d.columns if not c.
  startswith("_")]
  total = d.count()
  distinct_rows = d.select(*cols).distinct().count()
  print(f"\n  {t}  |  rows = {total}  |  duplicate rows = {total - distinct_rows}")
  stats = d.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__null")for c in cols],
    *[F.sum((F.trim(F.col(c)) == "").cast("int")).alias(f"{c}__empty") for c in cols],
  ).collect()[0].asDict()
  issues = {k: v for k, v in stats.items() if v and v > 0}
  print("nulls/empties:", issues if issues else "none")


  account  |  rows = 4500  |  duplicate rows = 0
nulls/empties: none

  card  |  rows = 892  |  duplicate rows = 0
nulls/empties: none

  client  |  rows = 5369  |  duplicate rows = 0
nulls/empties: none

  disp  |  rows = 5369  |  duplicate rows = 0
nulls/empties: none

  district  |  rows = 77  |  duplicate rows = 0
nulls/empties: none

  loan  |  rows = 682  |  duplicate rows = 0
nulls/empties: none

  order  |  rows = 6471  |  duplicate rows = 0
nulls/empties: {'k_symbol__empty': 1379}

  trans  |  rows = 1056320  |  duplicate rows = 0
nulls/empties: {'operation__null': 183114, 'k_symbol__null': 481881, 'bank__null': 782812, 'account__null': 760931, 'k_symbol__empty': 53433}


silver


In [0]:
silver_base = "/Volumes/workspace/default/lakehouse/silver"

def read_bronze(name):
  return spark.read.format("delta").load(f"{bronze_base}/{name}")

def clean_blanks(df):
  for c in df.columns:
    if not c.startswith("_"):
      df = df.withColumn(c, F.when(F.trim(F.col(c)).isin("","?"), None)
      .otherwise(F.trim(F.col(c))))
  return df

def to_date_yymmdd(col):
  return F.to_date(F.concat(F.lit("19"), F.substring(F.col(col), 1 ,6)) ,"yyyyMMdd")

def save_silver(df, name):
  df.write.format("delta").mode("overwrite").save(f"{silver_base}/{name}")
  n = spark.read.format("delta").load(f"{silver_base}/{name}").count()
  print(name, n , "rows")

In [0]:
account = clean_blanks(read_bronze("account"))

account = (account
           .withColumn("account_id", F.col("account_id").cast("int"))
           .withColumn("district_id", F.col("district_id").cast("int"))
           .withColumn("created_date", to_date_yymmdd("date")).drop("date")
           .replace({"POPLATEK MESICNE": "MONTHLY",
                     "POPLATEK PO OBRATU":"AFTER_TRANSACTION",
                     "POPLATEK TYDNE":"WEEKLY"}, subset=["frequency"]).dropDuplicates(["account_id"])
           )
save_silver(account, "account")


account 4500 rows


In [0]:
card = clean_blanks(read_bronze("card"))

card = (card
        .withColumn("card_id",F.col("card_id").cast("int"))
        .withColumn("disp_id",F.col("disp_id").cast("int"))
        .withColumn("issue_date", to_date_yymmdd("issued")).drop("issued")
        .dropDuplicates(["card_id"])
        )
save_silver(card, "card")

card 892 rows


In [0]:
client = clean_blanks(read_bronze("client"))
mm = F.substring("birth_number",3,2).cast("int")

client = (client
          .withColumn("client_id", F.col("client_id").cast("int"))
          .withColumn("district_id", F.col("district_id").cast("int"))
          .withColumn("birth_month", F.when(mm > 50 , mm - 50).otherwise(mm))
          .withColumn("birth_date", F.to_date(F.concat(
              F.lit("19"),
              F.substring("birth_number", 1,2),
              F.lpad("birth_month", 2, 0),
              F.substring("birth_number",5,2)) , "yyyyMMdd"))
          .drop("birth_month")
          .dropDuplicates(["client_id"]))
save_silver(client, "client")

client 5369 rows


In [0]:
disp = clean_blanks(read_bronze("disp"))

disp = (disp
        .withColumn("disp_id", F.col("disp_id").cast("int"))
        .withColumn("client_id", F.col("client_id").cast("int"))
        .withColumn("account_id", F.col("account_id").cast("int"))
        .dropDuplicates(["disp_id"])
        )
save_silver(disp, "disp")

disp 5369 rows


In [0]:
district = clean_blanks(read_bronze("district"))
names = {"A1":"district_id", "A2":"district_name", "A3":"region",
        "A4":"inhabitants", "A5":"muncipalities_below_500",
        "A6":"muncipalities_500_1999", "A7":"munncipalities_2000_9999",
        "A8":"muncipalities_above_10000", "A9":"cities", "A10":"urban_ratio",
        "A11":"avg_salary","A12":"unemployment_rate_95", "A13":"umemployment_rate_96",
        "A14":"entrepreneurs_per_1000", "A15":"crimes_95", "A16":"crimes_96"}
for old, new in names.items():
  district = district.withColumnRenamed(old, new)

int_cols =["district_id","inhabitants","muncipalities_below_500","muncipalities_500_1999",
           "munncipalities_2000_9999","muncipalities_above_10000","cities","avg_salary","crimes_95","crimes_96"]

dbl_cols =["urban_ratio","unemployment_rate_95","umemployment_rate_96","entrepreneurs_per_1000"]

for c in int_cols:
  district= district.withColumn(c, F.col(c).cast("int"))
for c in dbl_cols:
  district = district.withColumn(c, F.col(c).cast("double"))
district = district.dropDuplicates(["district_id"])
save_silver(district, "district")

district 77 rows


In [0]:
loan = clean_blanks(read_bronze("loan"))
loan = (loan
    .withColumn("loan_id", F.col("loan_id").cast("int"))
    .withColumn("account_id", F.col("account_id").cast("int"))
    .withColumn("loan_date", to_date_yymmdd("date"))
    .drop("date")
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn("duration", F.col("duration").cast("int"))
    .withColumn("payments", F.col("payments").cast("decimal(12,2)"))
    .withColumn("status_desc", F.col("status"))
    .replace({"A": "FINISHED_OK", "B": "FINISHED_NOT_PAID",
              "C": "RUNNING_OK", "D": "RUNNING_IN_DEBT"}, subset=["status_desc"])
    .dropDuplicates(["loan_id"]))
save_silver(loan, "loan")

loan 682 rows


In [0]:
order = clean_blanks(read_bronze("order"))
order = (order
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("account_id", F.col("account_id").cast("int"))
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .replace({"POJISTNE": "INSURANCE", "SIPO": "HOUSEHOLD",
              "LEASING": "LEASING", "UVER": "LOAN_PAYMENT"}, subset=["k_symbol"])
    .dropDuplicates(["order_id"]))
save_silver(order, "order")

order 6471 rows


In [0]:
trans = clean_blanks(read_bronze("trans"))
trans = (trans
    .withColumn("trans_id", F.col("trans_id").cast("int"))
    .withColumn("account_id", F.col("account_id").cast("int"))
    .withColumn("txn_date", to_date_yymmdd("date"))
    .drop("date")
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn("balance", F.col("balance").cast("decimal(12,2)"))
    .withColumnRenamed("bank", "partner_bank")
    .withColumnRenamed("account", "partner_account")
    .replace({"PRIJEM": "CREDIT", "VYDAJ": "WITHDRAWAL",
              "VYBER": "CASH_WITHDRAWAL"}, subset=["type"])
    .replace({"VYBER KARTOU": "CARD_WITHDRAWAL", "VKLAD": "CASH_CREDIT",
              "PREVOD Z UCTU": "TRANSFER_IN", "VYBER": "CASH_WITHDRAWAL",
              "PREVOD NA UCET": "TRANSFER_OUT"}, subset=["operation"])
    .replace({"POJISTNE": "INSURANCE", "SLUZBY": "SERVICE_FEE",
              "UROK": "INTEREST_CREDITED", "SANKC. UROK": "PENALTY_INTEREST",
              "SIPO": "HOUSEHOLD", "DUCHOD": "PENSION",
              "UVER": "LOAN_PAYMENT"}, subset=["k_symbol"])
    .dropDuplicates(["trans_id"]))
save_silver(trans, "trans")

trans 1056320 rows


In [0]:
cr = spark.read.format("delta").load(f"{silver_base}/client")
print("client birth_date null:", cr.filter(F.col("birth_date").isNull()).count())

tr = spark.read.format("delta").load(f"{silver_base}/trans")
print("trans txn_date null:", tr.filter(F.col("txn_date").isNull()).count())

t=  spark.read.format("delta").load(f"{silver_base}/trans")
for c in ["type","operation","k_symbol"]:
  t.groupBy(c).count().orderBy(F.desc("count")).show(truncate=False)
t.select(F.min("txn_date"), F.max("txn_date")).show()
t.printSchema()



client birth_date null: 0
trans txn_date null: 0
+---------------+------+
|type           |count |
+---------------+------+
|WITHDRAWAL     |634571|
|CREDIT         |405083|
|CASH_WITHDRAWAL|16666 |
+---------------+------+

+---------------+------+
|operation      |count |
+---------------+------+
|CASH_WITHDRAWAL|434918|
|TRANSFER_OUT   |208283|
|NULL           |183114|
|CASH_CREDIT    |156743|
|TRANSFER_IN    |65226 |
|CARD_WITHDRAWAL|8036  |
+---------------+------+

+-----------------+------+
|k_symbol         |count |
+-----------------+------+
|NULL             |535314|
|INTEREST_CREDITED|183114|
|SERVICE_FEE      |155832|
|HOUSEHOLD        |118065|
|PENSION          |30338 |
|INSURANCE        |18500 |
|LOAN_PAYMENT     |13580 |
|PENALTY_INTEREST |1577  |
+-----------------+------+

+-------------+-------------+
|min(txn_date)|max(txn_date)|
+-------------+-------------+
|   1993-01-01|   1998-12-31|
+-------------+-------------+

root
 |-- trans_id: integer (nullable = true)
 |

In [0]:
tr = spark.read.format("delta").load(f"{silver_base}/trans")
tr.filter(F.col("operation").isNull()).groupBy("k_symbol").count().show()

+-----------------+------+
|         k_symbol| count|
+-----------------+------+
|INTEREST_CREDITED|183114|
+-----------------+------+



Gold


In [0]:
gold_base ="/Volumes/workspace/default/lakehouse/gold"

def read_silver(name):
  return spark.read.format("delta").load(f"{silver_base}/{name}")

def save_gold(df,name):
  df.write.format("delta").mode("overwrite").save(f"{gold_base}/{name}")
  print(name, spark.read.format("delta").load(f"{gold_base}/{name}").count(), "rows")

In [0]:
trans = read_silver("trans")

monthly =( trans
           .withColumn("month",  F.date_format("txn_date","yyyy-MM"))
           .groupBy("month")
           .agg(
               F.count("*").alias("txn_count"),
               F.sum(F.when(F.col("type")=="CREDIT", F.col("amount")).otherwise(0)).alias("total_credit"),
               F.sum(F.when(F.col("type") != "CREDIT", F.col("amount")).otherwise(0)).alias("total_debit")
               )
           .withColumn("net_flow", F.col("total_credit") - F.col("total_debit"))
           .orderBy("month")
          )

save_gold(monthly, "monthly_cashflow")
monthly.show(5)

monthly_cashflow 72 rows
+-------+---------+------------+-----------+----------+
|  month|txn_count|total_credit|total_debit|  net_flow|
+-------+---------+------------+-----------+----------+
|1993-01|      177|   667457.60|   34700.00| 632757.60|
|1993-02|      395|  2445498.70|  281426.60|2164072.10|
|1993-03|      676|  3855629.40|  874689.20|2980940.20|
|1993-04|      913|  5325748.80| 2052619.00|3273129.80|
|1993-05|     1306|  7664727.40| 4016026.00|3648701.40|
+-------+---------+------------+-----------+----------+
only showing top 5 rows


In [0]:
account = read_silver("account")
district = read_silver("district")

per_account= (trans.groupBy("account_id").agg(
    F.count("*").alias("txn_count"),
    F.sum(F.when(F.col("type") == "CREDIT", F.col("amount")).otherwise(0)).alias("total_credit"),
    F.sum(F.when(F.col("type") != "CREDIT", F.col("amount")).otherwise(0)).alias("total_debit"),
    F.round(F.avg("balance"),2).alias("avg_balance"))
)

account_summary = (account
                   .join(per_account, on="account_id", how= "left")
                   .join(district.select("district_id","district_name","region"), "district_id", how="left")
                   .select("account_id","district_name","region","frequency","created_date","txn_count","total_credit","total_debit","avg_balance")
                   )
save_gold(account_summary,"account_summary")
account_summary.show(5)

account_summary 4500 rows
+----------+---------------+-------------+---------+------------+---------+------------+-----------+-----------+
|account_id|  district_name|       region|frequency|created_date|txn_count|total_credit|total_debit|avg_balance|
+----------+---------------+-------------+---------+------------+---------+------------+-----------+-----------+
|      1926|     Litomerice|north Bohemia|  MONTHLY|  1993-01-06|      447|   418083.90|  395886.20|   19960.93|
|      2122|Frydek - Mistek|north Moravia|  MONTHLY|  1993-01-08|      470|  2796288.00| 2754483.80|   60335.07|
|      4034|    Hl.m. Praha|       Prague|  MONTHLY|  1993-01-18|      358|   541310.40|  466421.60|   38972.21|
|       280|        Klatovy| west Bohemia|  MONTHLY|  1993-01-23|      481|   979304.40|  913785.60|   32434.17|
|       443|           Zlin|south Moravia|  MONTHLY|  1993-01-28|      455|   223792.10|  201561.60|   22156.90|
+----------+---------------+-------------+---------+------------+-----

In [0]:
loan = read_silver("loan")

loan_region = (loan
    .join(account.select("account_id", "district_id"), "account_id")
    .join(district.select("district_id", "region"), "district_id"))

loan_summary = (loan_region.groupBy("region", "status_desc").agg(
        F.count("*").alias("loans"),
        F.sum("amount").alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"))
    .orderBy("region", "status_desc"))

save_gold(loan_summary, "loan_summary_by_region")
loan_summary.show(10, truncate=False)

loan_summary_by_region 30 rows
+---------------+-----------------+-----+------------+----------+
|region         |status_desc      |loans|total_amount|avg_amount|
+---------------+-----------------+-----+------------+----------+
|Prague         |FINISHED_NOT_PAID|3    |655020.00   |218340.00 |
|Prague         |FINISHED_OK      |34   |2985540.00  |87810.00  |
|Prague         |RUNNING_IN_DEBT  |4    |1372116.00  |343029.00 |
|Prague         |RUNNING_OK       |43   |7919736.00  |184179.91 |
|central Bohemia|FINISHED_NOT_PAID|5    |798348.00   |159669.60 |
|central Bohemia|FINISHED_OK      |26   |2347260.00  |90279.23  |
|central Bohemia|RUNNING_IN_DEBT  |5    |1214652.00  |242930.40 |
|central Bohemia|RUNNING_OK       |54   |9625044.00  |178241.56 |
|east Bohemia   |FINISHED_OK      |19   |1734240.00  |91275.79  |
|east Bohemia   |RUNNING_IN_DEBT  |9    |1670004.00  |185556.00 |
+---------------+-----------------+-----+------------+----------+
only showing top 10 rows


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.berka_gold")

for t in ["monthly_cashflow", "account_summary", "loan_summary_by_region"]:
    df = spark.read.format("delta").load(f"{gold_base}/{t}")
    df.write.mode("overwrite").saveAsTable(f"workspace.berka_gold.{t}")
    print(t, "loaded")

monthly_cashflow loaded
account_summary loaded
loan_summary_by_region loaded


In [0]:
display(spark.sql("""
    SELECT region, status_desc, loans, total_amount
    FROM workspace.berka_gold.loan_summary_by_region
    ORDER BY total_amount DESC
    LIMIT 10
"""))

region,status_desc,loans,total_amount
south Moravia,RUNNING_OK,81,13941936.00
north Moravia,RUNNING_OK,64,10962444.00
east Bohemia,RUNNING_OK,56,10539480.00
central Bohemia,RUNNING_OK,54,9625044.00
Prague,RUNNING_OK,43,7919736.00
south Bohemia,RUNNING_OK,33,5877972.00
north Bohemia,RUNNING_OK,41,5694444.00
west Bohemia,RUNNING_OK,31,4517316.00
north Moravia,FINISHED_OK,35,3247776.00
north Moravia,RUNNING_IN_DEBT,12,3082944.00


In [0]:
display(spark.sql("""
    SELECT month, total_credit, total_debit, net_flow
    FROM workspace.berka_gold.monthly_cashflow
    ORDER BY month
"""))

month,total_credit,total_debit,net_flow
1993-01,667457.60,34700.00,632757.60
1993-02,2445498.70,281426.60,2164072.10
1993-03,3855629.40,874689.20,2980940.20
1993-04,5325748.80,2052619.00,3273129.80
1993-05,7664727.40,4016026.00,3648701.40
1993-06,10769273.40,7522675.50,3246597.90
1993-07,10636458.50,7384836.20,3251622.30
1993-08,11957341.00,9071470.60,2885870.40
1993-09,13515566.70,9923134.90,3592431.80
1993-10,15482959.80,12087939.90,3395019.90
